# Anatomy of Effective Prompts

Every LLM response is shaped by the text you send. Before learning advanced techniques like few-shot examples or chain-of-thought, you need a clear picture of **what belongs in a prompt** and **how to arrange it**. Weak prompts are often vague, unstructured, or contradictory. Strong prompts name the task, supply necessary context, define the desired output, and state constraints explicitly.

This notebook dissects the building blocks of effective prompts: the five core components, how chat APIs map them to message roles, principles for writing clear instructions, structural patterns (including delimiters), side-by-side weak vs strong examples, and practical token awareness. A live API demonstration shows how structure and specificity change real model outputs.

Read **01. Introduction to Prompt Engineering** first. Later notebooks apply these components to zero-shot, few-shot, roles, and structured output.

---

## 1. The Five Components of a Prompt

Most useful prompts — whether a single user message or a multi-message chat — combine up to five elements. Not every task needs all five, but knowing the list helps you spot what is missing when output quality drops.

| Component | Purpose | Example phrase |
|-----------|---------|----------------|
| **Instruction** | What the model should *do* | "Summarize the text below in three bullets." |
| **Context** | Background the model needs | Product docs, meeting notes, user profile |
| **Input** | The data to process | The article, ticket, code snippet, question |
| **Output format** | Shape of the answer | "JSON with keys `title`, `summary`", "markdown table" |
| **Constraints** | Limits and rules | Max length, tone, what to avoid, language |

### 1.1 Instruction

The instruction is the **verb** of the prompt: summarize, classify, translate, extract, compare, debug. Without a clear instruction, the model defaults to open-ended continuation — it guesses what kind of answer you wanted.

Weak: *"Here is some text about our refund policy."* (no task)

Strong: *"Summarize the refund policy below for a customer support agent. Highlight exceptions in bold."*

### 1.2 Context

Context is **static or semi-static background** that frames the task: domain knowledge, definitions, prior decisions, or style guides. In RAG systems, retrieved documents become context inserted before the user question.

Context should be **relevant**. Irrelevant paragraphs add tokens, increase cost, and can distract the model (a phenomenon sometimes called "lost in the middle" for very long contexts).

### 1.3 Input

Input is the **variable payload** — the thing that changes every request. In a support bot, the input is the customer's message; in batch processing, each row of data.

Separate input from instructions and context using headings or delimiters (Section 2) so the model knows which text to transform.

### 1.4 Output Format

Specify how the answer should look: bullet list, table, JSON, single word label, step numbering. Format instructions reduce post-processing and parse errors.

If your application expects machine-readable output, say so explicitly — covered in depth in **06. Structured Output and Format Control**.

### 1.5 Constraints

Constraints bound behavior:

- **Length** — "under 100 words", "exactly five items"
- **Tone** — formal, friendly, technical
- **Scope** — "only use information from the context; do not invent facts"
- **Negatives** — "do not mention pricing", "no code outside Python"

Constraints conflict if you ask for both "exhaustive detail" and "one sentence." Resolve conflicts before calling the API.

### 1.6 Mapping Components to Chat API Messages

Provider APIs represent prompts as a **list of messages** with roles. A typical mapping:

| Component | Where it usually lives |
|-----------|------------------------|
| Role + global rules | `system` message |
| Context + input + instruction | `user` message (or split across turns) |
| Prior answers | `assistant` messages in history |

**Example structure:**

```
system:  You are a technical editor. Use concise language. Never invent citations.
user:    Instruction: Improve clarity of the paragraph below.
         Context: Audience is junior developers.
         Constraints: Keep under 120 words. Preserve meaning.
         ---
         Input:
         {paragraph text here}
```

Claude uses a separate `system` parameter; Gemini uses `system_instruction` in config. The **components** are the same even when the API shape differs — see **03. LLM API Integration** provider notebooks.

---

## 2. Writing Clear Instructions

Clear instructions answer four questions for the model:

1. **What** task to perform
2. **Who** the audience is (if relevant)
3. **What** a good result looks like
4. **What** to do when information is missing

### 2.1 Specificity and Scope

| Vague | Specific |
|-------|----------|
| "Fix this email." | "Rewrite the email to sound polite and under 150 words. Keep the meeting time unchanged." |
| "Analyze sales." | "List the top three revenue risks from the Q3 figures below. One sentence each." |
| "Is this good code?" | "Review the function for bugs and readability. Return: BUGS (list), SUGGESTIONS (list)." |

Scope creep happens when the instruction invites unlimited work ("explain everything about Kubernetes"). Narrow the domain, depth, and format.

### 2.2 Success Criteria

When possible, state what success means:

- "If the text does not mention a date, respond with `DATE_NOT_FOUND`."
- "Each bullet must cite a line number from the input."
- "Answer only yes or no."

Explicit success criteria make outputs testable — important for **09. Prompt Evaluation and Iteration**.

### 2.3 Delimiters and Structure

Long prompts mix instructions, context, and user data. **Delimiters** mark boundaries so the model (and you) can tell sections apart.

Common patterns:

| Delimiter | Use case |
|-----------|----------|
| `---` or `###` | Section headers in markdown |
| Triple quotes `\"\"\"` | Wrap multi-line input |
| XML tags `<document>...</document>` | Popular in production templates |
| `INPUT:` / `OUTPUT:` labels | Simple key-value sections |

**Why delimiters help:** They reduce the chance the model treats instruction text as data to summarize, or vice versa. They also help when user input contains phrases that look like instructions (prompt injection — handled in depth in Advanced Topics).

### 2.4 Order of Sections

A reliable default order inside the user message:

1. Instruction (what to do)
2. Context (background)
3. Constraints (rules)
4. Output format
5. Delimiter
6. Input (data to process)

Putting **input last** mirrors how humans read: task first, then evidence. Some extraction tasks place examples before input (few-shot — next notebook).

---

## 3. Good vs Weak Prompts

The same model with the same parameters can succeed or fail based on prompt anatomy alone. Below are recurring **weak patterns** and **fixes**.

| Weak pattern | Problem | Fix |
|--------------|---------|-----|
| No explicit task | Model chatters or guesses format | Start with an imperative instruction |
| Buried instruction | Model focuses on wrong section | Put instruction at top; delimiter before input |
| Missing output format | Unparseable free text | Specify bullets, JSON, labels |
| Contradictory rules | Inconsistent behavior | Remove conflicts; prioritize one goal |
| Huge context, tiny question | Cost + distraction | Trim context; retrieve only relevant chunks |
| Assuming shared knowledge | Wrong audience level | State audience and definitions |

### 3.1 Worked Example — Customer Email Classification

**Weak prompt:**

```
We get lots of emails. This one is from a customer. What should we do?
Subject: Charger stopped working
Body: I bought the X200 last month and it won't charge.
```

Problems: no categories, no output shape, no definition of "what we should do."

**Strong prompt:**

```
Classify the support email below into exactly one category:
REFUND | REPLACEMENT | TECH_SUPPORT | BILLING | OTHER

Reply with only the category label in uppercase.
If unsure, reply OTHER.

---
EMAIL:
Subject: Charger stopped working
Body: I bought the X200 last month and it won't charge.
```

The strong version includes **instruction**, **closed set of labels**, **output format**, **fallback rule**, **delimiter**, and **input**.

---

## 4. Demonstration: Weak vs Structured Prompt

The code below sends the same customer email through two prompts:

1. **Weak** — informal, no structure
2. **Structured** — all five components with delimiters

Replace the placeholder API key before running.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4o-mini"

EMAIL = """Subject: Charger stopped working
Body: I bought the X200 last month and it won't charge. Order #88421.
I'd like this resolved quickly."""

weak_user = f"""We get lots of emails. This one is from a customer. What should we do?

{EMAIL}"""

structured_user = f"""Instruction: Classify the support email into exactly one category.
Categories: REFUND | REPLACEMENT | TECH_SUPPORT | BILLING | OTHER

Context: X200 is a wireless charger sold by our electronics store.

Output format: Reply with only the category label in uppercase on the first line,
then one short sentence explaining why (max 20 words).

Constraints: Use only information in the email. If unsure, use OTHER.

---
EMAIL:
{EMAIL}"""

SYSTEM = "You are a support ticket classifier. Follow the user's format exactly."


def complete(user_prompt: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
        max_tokens=120,
    )
    return response.choices[0].message.content or ""


print("=== WEAK PROMPT OUTPUT ===")
print(complete(weak_user))
print("\n" + "=" * 50 + "\n")
print("=== STRUCTURED PROMPT OUTPUT ===")
print(complete(structured_user))

### 4.1 What to Observe

- The **weak** prompt often produces a long, conversational plan instead of a single label.
- The **structured** prompt should return a category first, then a brief reason — easier to route in code (`if label == "REPLACEMENT": ...`).
- `temperature=0.0` keeps comparisons stable when you re-run.

Try adding XML tags instead of `---`:

```
<email>{EMAIL}</email>
```

and note whether parsing consistency improves for your model.

---

## 5. Demonstration: All Components in One Template

The next cell uses a reusable **prompt template** with placeholders. In production you would substitute `{context}` and `{input}` per request — the pattern used in **10. Production Prompt Management**.

In [ ]:
PROMPT_TEMPLATE = """Instruction: {instruction}

Context:
{context}

Output format:
{output_format}

Constraints:
{constraints}

---
Input:
{input}
"""

user_message = PROMPT_TEMPLATE.format(
    instruction="Extract action items from the meeting notes.",
    context="Team: backend squad. Sprint 14 planning.",
    output_format="Numbered list. Each item starts with a verb. Max 5 items.",
    constraints="Only include explicit commitments. Do not invent tasks.",
    input="""Alice: I'll fix the login timeout bug by Friday.
Bob: API docs need updating after the auth change.
Carol: We should probably look at caching someday.""",
)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a precise meeting assistant."},
        {"role": "user", "content": user_message},
    ],
    temperature=0.2,
    max_tokens=200,
)

print(user_message)
print("\n" + "=" * 50 + "\nMODEL OUTPUT:\n")
print(response.choices[0].message.content)

Notice how Carol's vague "someday" comment should be **excluded** because the constraint says "explicit commitments only." If the model includes it, tighten the constraint or add a negative example (few-shot — next notebook).

---

## 6. Token and Context Awareness

Every word in instruction, context, and input consumes **input tokens** and competes for space in the **context window**. Prompt engineering is not "write as much as possible" — it is "write what is necessary."

### 6.1 Costs of Verbosity

- Higher API cost (priced per token)
- Higher latency (more tokens to process)
- Risk of pushing out room for the **completion** you need
- Risk of burying the actual question in noise

### 6.2 Practical Guidelines

| Practice | Rationale |
|----------|-----------|
| Move stable rules to `system` once | Reuse without repeating in every user turn |
| Trim redundant politeness | "Please" stacks add tokens without changing behavior |
| Summarize long context when possible | Pre-summarize docs before Q&A |
| Preflight token counts | Use tiktoken or provider APIs before large batches |

See **02. GenAI & LLM Fundamentals — Tokens and Context Windows** for counting and budgeting. The equation from the intro notebook still applies:

$$\text{available for output} \approx \text{context window} - \text{prompt tokens} - \text{safety margin}$$

### 6.3 When Longer Prompts Are Worth It

Longer prompts are justified when:

- **Few-shot examples** materially improve accuracy
- **Retrieved documents** are required for correct answers (RAG)
- **Detailed policies** must be in context for compliance

The skill is knowing when extra tokens buy measurable quality — not adding text by default.

---

## 7. Checklist Before You Call the API

Use this quick checklist when drafting a new prompt:

1. Is there a clear **instruction** (verb + task)?
2. Is **context** included — and only what is needed?
3. Is **input** separated from instructions (delimiter)?
4. Is **output format** specified?
5. Are **constraints** consistent and testable?
6. Does the **system** message carry stable role/rules?
7. Is **temperature** appropriate (low for extraction, higher for creativity)?
8. Have you tested on **more than one** input?

Printing the filled template (as in Section 5) before sending helps catch missing sections early.

---

## 8. Summary

Effective prompts combine **instruction**, **context**, **input**, **output format**, and **constraints**. Chat APIs map these onto **system** and **user** messages; provider details differ, but the anatomy is the same.

Clear instructions are **specific**, scoped, and backed by **success criteria**. **Delimiters** and consistent section order keep instructions separate from data. Weak prompts fail from vagueness, missing format, or contradiction — not from model incapacity.

Be **token-aware**: every section has a cost. Trim noise; keep examples and context that measurably help.

The demonstrations showed how structure changes outputs from open-ended prose to routable labels and actionable lists. Next: **03. Zero-Shot and Few-Shot Prompting** — when to rely on instructions alone and when to add examples in the prompt.